In [ ]:
from __future__ import annotations

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq


PERIOD = 'Full'  # 'Full', 'Early' or 'Late'
PARQUET_PATH = 'interactions1.parquet'
USERS_PATH = 'users1.parquet'
OUTPUT_DIR = Path('.')
BOOTSTRAP_REPS = 200
BATCH_SIZE = 250_000
RANDOM_SEED = 42

PERIOD = PERIOD.strip().title()
if PERIOD not in {'Full', 'Early', 'Late'}:
    raise ValueError("PERIOD must be 'Full', 'Early', or 'Late'")

INTERACTIONS_PATH = Path(PARQUET_PATH)
USERS_FILE = Path(USERS_PATH)
if not INTERACTIONS_PATH.exists():
    raise FileNotFoundError(INTERACTIONS_PATH)
if not USERS_FILE.exists():
    raise FileNotFoundError(USERS_FILE)

interaction_schema = pq.ParquetFile(INTERACTIONS_PATH).schema_arrow
user_schema = pq.ParquetFile(USERS_FILE).schema_arrow
print('INTERACTIONS schema (metadata only):')
print(interaction_schema)
print('USERS schema (metadata only):')
print(user_schema)
print('Interaction columns/types:', [(f.name, f.type) for f in interaction_schema])
print('User columns/types:', [(f.name, f.type) for f in user_schema])

required_interaction = {'day', 'source', 'target'}
required_user = {'day', 'user_id'}
if not required_interaction.issubset(set(interaction_schema.names)):
    raise ValueError(f'Missing interaction columns: {required_interaction - set(interaction_schema.names)}')
if not required_user.issubset(set(user_schema.names)):
    raise ValueError(f'Missing user columns: {required_user - set(user_schema.names)}')

# Small sample: does not materialize the complete dataset.
sample_table = pq.ParquetFile(INTERACTIONS_PATH).read_row_group(0, columns=['day', 'source', 'target'], use_threads=True)
print('Sample rows:')
display(sample_table.slice(0, min(5_000, sample_table.num_rows)).to_pandas())
del sample_table
gc.collect()

INTERACTIONS schema (metadata only):
day: timestamp[us]
source: large_string
target: large_string
interaction_id: large_string
post_id: large_string
interaction_type: large_string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 792
USERS schema (metadata only):
day: timestamp[us]
user_id: large_string
source_type: large_string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 439
Interaction columns/types: [('day', TimestampType(timestamp[us])), ('source', DataType(large_string)), ('target', DataType(large_string)), ('interaction_id', DataType(large_string)), ('post_id', DataType(large_string)), ('interaction_type', DataType(large_string))]
User columns/types: [('day', TimestampType(timestamp[us])), ('user_id', DataType(large_string)), ('source_type', DataType(large_string))]
Esempio di righe:


,day,source,target
0,2026-01-28,08731503-35fc-4c05-a9c1-57b9633eba54,9939911b-bc65-407a-ae67-671afb8c0148
1,2026-01-28,08731503-35fc-4c05-a9c1-57b9633eba54,cfb773d9-d8df-4b4b-80ff-af36dbb3bbaa
2,2026-01-28,08731503-35fc-4c05-a9c1-57b9633eba54,d57b3ba7-7a4f-4b74-8a6e-b31d0addebf3
3,2026-01-28,08731503-35fc-4c05-a9c1-57b9633eba54,d57b3ba7-7a4f-4b74-8a6e-b31d0addebf3
4,2026-01-28,08731503-35fc-4c05-a9c1-57b9633eba54,d57b3ba7-7a4f-4b74-8a6e-b31d0addebf3
...,...,...,...
4995,2026-01-30,1f4c4f4e-141c-45ae-92f9-b120272f224c,de3976c9-a129-4c80-a687-51aa6a1d7169
4996,2026-01-30,1f4c4f4e-141c-45ae-92f9-b120272f224c,e21c89e0-dcb3-4524-bf79-7dbe1e5f2c60
4997,2026-01-30,1f4c4f4e-141c-45ae-92f9-b120272f224c,e96cd5ac-c14e-4027-9020-e1e30b2877cb
4998,2026-01-30,1f4c4f4e-141c-45ae-92f9-b120272f224c,e9e40097-6110-4fe1-abfc-81c68cdfa6f0


3

In [ ]:
def batches(path: Path, columns: list[str]):
    dataset = ds.dataset(path, format='parquet')
    return dataset.to_batches(columns=columns, batch_size=BATCH_SIZE, use_threads=True)


# Each Parquet file already contains only the period specified by PERIOD.
# All rows are read, without filtering the day column.
user_ids: list[pd.Series] = []
for batch in batches(USERS_FILE, ['user_id']):
    frame = batch.to_pandas()
    valid = frame['user_id'].notna()
    if valid.any():
        user_ids.append(frame.loc[valid, 'user_id'])

if user_ids:
    users_period = pd.concat(user_ids, ignore_index=True).drop_duplicates()
else:
    users_period = pd.Series(dtype='string')
del user_ids
# The user file may contain multiple rows for the same user: the node universe uses distinct IDs.
gc.collect()

unique_edge_chunks: list[pd.DataFrame] = []
strength_chunks: list[pd.Series] = []
for batch in batches(INTERACTIONS_PATH, ['source', 'target']):
    frame = batch.to_pandas()
    valid = frame['source'].notna() & frame['target'].notna()
    frame = frame.loc[valid, ['source', 'target']]
    if frame.empty:
        continue
    unique_edge_chunks.append(frame.drop_duplicates(ignore_index=True))
    # Unit weight: one row = one received interaction.
    strength_chunks.append(frame.groupby('target', sort=False).size())

if unique_edge_chunks:
    edges = pd.concat(unique_edge_chunks, ignore_index=True).drop_duplicates(ignore_index=True)
else:
    edges = pd.DataFrame({'source': pd.Series(dtype='string'), 'target': pd.Series(dtype='string')})
if strength_chunks:
    in_strength_series = pd.concat(strength_chunks).groupby(level=0, sort=False).sum()
else:
    in_strength_series = pd.Series(dtype='int64')
del unique_edge_chunks, strength_chunks, frame
gc.collect()

node_index = pd.Index(pd.unique(pd.concat([users_period, edges['source'], edges['target']], ignore_index=True)))
in_degree_series = edges.groupby('target', sort=False)['source'].nunique()
out_degree_series = edges.groupby('source', sort=False)['target'].nunique()
in_degree = in_degree_series.reindex(node_index, fill_value=0).to_numpy(dtype=np.int64, copy=False)
out_degree = out_degree_series.reindex(node_index, fill_value=0).to_numpy(dtype=np.int64, copy=False)
in_strength = in_strength_series.reindex(node_index, fill_value=0).to_numpy(dtype=np.int64, copy=False)

metric_values = {'In-degree': in_degree, 'Out-degree': out_degree, 'In-strength': in_strength}
summary = pd.DataFrame({
    'Metric': list(metric_values),
    'N_nodes': [len(v) for v in metric_values.values()],
    'N_positive': [int(np.count_nonzero(v > 0)) for v in metric_values.values()],
    'Mean': [float(np.mean(v)) if len(v) else np.nan for v in metric_values.values()],
    'Median': [float(np.median(v)) if len(v) else np.nan for v in metric_values.values()],
    'Min': [int(np.min(v)) if len(v) else np.nan for v in metric_values.values()],
    'Max': [int(np.max(v)) if len(v) else np.nan for v in metric_values.values()],
})
print(f'Nodes in period {PERIOD}: {len(node_index):,}; distinct directed edges: {len(edges):,}')
display(summary)
print('Check: N_positive cannot exceed N_nodes; Min must be 0 or greater.')
assert (summary['N_positive'] <= summary['N_nodes']).all()
assert (summary['Min'] >= 0).all()

Nodi nel periodo Full: 166,668; archi diretti distinti: 1,028,465


,Metric,N_nodes,N_positive,Mean,Median,Min,Max
0,In-degree,166668,58560,6.170741,0.0,0,10126
1,Out-degree,166668,28576,6.170741,0.0,0,21520
2,In-strength,166668,58560,71.131939,0.0,0,521166


Controllo: N_positive non puo superare N_nodes; Min deve essere 0 o maggiore.


## Fitting and bootstrap

The reported KS statistic is the goodness-of-fit measure for the power law. Comparisons with the lognormal, exponential, and truncated power law use only `distribution_compare`, that is, the likelihood-ratio test; `R > 0` favors the power law and `R < 0` favors the alternative.

In [ ]:
try:
    import powerlaw
except ImportError as exc:
    raise ImportError("Install the package with: pip install powerlaw") from exc


def ks_statistic(fit) -> float:
    # powerlaw 2.0.0 exposes D on the fit, while Distribution.KS() may fail.
    distance = getattr(fit, 'D', None)
    return float(distance) if distance is not None else np.nan


def safe_compare(fit, alternative: str) -> tuple[float, float]:
    try:
        ratio, p_value = fit.distribution_compare('power_law', alternative, normalized_ratio=False)
        return float(ratio), float(p_value)
    except Exception:
        return np.nan, np.nan


def bootstrap_powerlaw_p(values: np.ndarray, fit, repetitions: int, seed: int) -> float:
    rng = np.random.default_rng(seed)
    tail = values[values >= fit.xmin]
    body = values[values < fit.xmin]
    if len(tail) == 0 or repetitions <= 0:
        return np.nan
    observed_ks = ks_statistic(fit)
    exceed = 0
    for _ in range(repetitions):
        synthetic_tail = np.asarray(fit.power_law.generate_random(len(tail)), dtype=float)
        if len(body):
            synthetic_body = rng.choice(body, size=len(body), replace=True)
            synthetic = np.concatenate([synthetic_body, synthetic_tail])
        else:
            synthetic = synthetic_tail
        try:
            refit = powerlaw.Fit(synthetic, xmin=fit.xmin, estimate_discrete=False, verbose=False)
            synthetic_ks = ks_statistic(refit)
            exceed += synthetic_ks >= observed_ks
        except Exception:
            continue
    return float((exceed + 1) / (repetitions + 1))


def fit_metric(metric: str, values: np.ndarray, seed_offset: int) -> dict:
    positive = np.asarray(values, dtype=float)
    positive = positive[np.isfinite(positive) & (positive > 0)]
    result = {'Period': PERIOD, 'Metric': metric, 'N_positive': int(len(positive))}
    keys = ['xmin', 'alpha', 'alpha_se', 'n_tail', 'KS', 'bootstrap_p', 'PL_vs_lognormal_R', 'PL_vs_lognormal_p', 'PL_vs_exponential_R', 'PL_vs_exponential_p', 'PL_vs_truncated_R', 'PL_vs_truncated_p']
    result.update({key: np.nan for key in keys})
    if len(positive) < 2 or np.unique(positive).size < 2:
        return result
    fit = powerlaw.Fit(positive, discrete=True, verbose=False)
    result['xmin'] = float(fit.xmin)
    result['alpha'] = float(fit.alpha)
    result['alpha_se'] = float(getattr(fit, 'sigma', np.nan))
    result['n_tail'] = int(np.sum(positive >= fit.xmin))
    result['KS'] = ks_statistic(fit)
    result['bootstrap_p'] = bootstrap_powerlaw_p(positive, fit, BOOTSTRAP_REPS, RANDOM_SEED + seed_offset)
    for alternative, suffix in [('lognormal', 'lognormal'), ('exponential', 'exponential'), ('truncated_power_law', 'truncated')]:
        ratio, p_value = safe_compare(fit, alternative)
        result[f'PL_vs_{suffix}_R'] = ratio
        result[f'PL_vs_{suffix}_p'] = p_value
    return result


results = pd.DataFrame([fit_metric(name, values, index) for index, (name, values) in enumerate(metric_values.items())])
display(results)

c:\Moltbok\.venv\Lib\site-packages\powerlaw\distributions.py:776: UserWarning: No valid fits found for distribution lognormal.
  warnings.warn(f"No valid fits found for distribution {self.name}.")
c:\Moltbok\.venv\Lib\site-packages\powerlaw\distributions.py:808: UserWarning: Fitted parameters are very close to the edge of parameter ranges for distribution exponential; consider changing these ranges.
  warnings.warn(f'Fitted parameters are very close to the edge of parameter ranges for distribution {self.name}; consider changing these ranges.')
c:\Moltbok\.venv\Lib\site-packages\powerlaw\distributions.py:808: UserWarning: Fitted parameters are very close to the edge of parameter ranges for distribution truncated_power_law; consider changing these ranges.
  warnings.warn(f'Fitted parameters are very close to the edge of parameter ranges for distribution {self.name}; consider changing these ranges.')
c:\Moltbok\.venv\Lib\site-packages\powerlaw\distributions.py:808: UserWarning: Fitted par

,Period,Metric,N_positive,xmin,alpha,alpha_se,n_tail,KS,bootstrap_p,PL_vs_lognormal_R,PL_vs_lognormal_p,PL_vs_exponential_R,PL_vs_exponential_p,PL_vs_truncated_R,PL_vs_truncated_p
0,Full,In-degree,58560,147.0,2.913093,0.059152,1046,0.027620,0.004975,0.015470,6.837305e-01,298.053012,2.504120e-05,0.043614,0.767731
1,Full,Out-degree,28576,62.0,2.048877,0.021178,2453,0.016456,0.004975,-1.582337,2.696662e-01,1377.109329,1.190181e-24,-4.295738,0.003377
2,Full,In-strength,58560,6.0,1.555974,0.003116,31826,0.035990,0.004975,-79.194928,3.232744e-19,56146.290641,0.000000e+00,-213.360941,0.000000


In [ ]:
result_columns = [
    'Period', 'Metric', 'N_positive', 'xmin', 'alpha', 'alpha_se', 'n_tail', 'KS', 'bootstrap_p',
    'PL_vs_lognormal_R', 'PL_vs_lognormal_p',
    'PL_vs_exponential_R', 'PL_vs_exponential_p',
    'PL_vs_truncated_R', 'PL_vs_truncated_p',
]
results = results[result_columns]
csv_path = OUTPUT_DIR / f'powerlaw_results_{PERIOD.upper()}.csv'
xlsx_path = OUTPUT_DIR / f'powerlaw_results_{PERIOD.upper()}.xlsx'
results.to_csv(csv_path, index=False)
results.to_excel(xlsx_path, index=False)
display(results)
print(f'Only the final results were exported: {csv_path.resolve()} and {xlsx_path.resolve()}')

# Explicitly release large intermediate objects; the Parquet files were never written.
del edges, users_period, node_index, in_degree_series, out_degree_series, in_strength_series
gc.collect()

,Period,Metric,N_positive,xmin,alpha,alpha_se,n_tail,KS,bootstrap_p,PL_vs_lognormal_R,PL_vs_lognormal_p,PL_vs_exponential_R,PL_vs_exponential_p,PL_vs_truncated_R,PL_vs_truncated_p
0,Full,In-degree,58560,147.0,2.913093,0.059152,1046,0.027620,0.004975,0.015470,6.837305e-01,298.053012,2.504120e-05,0.043614,0.767731
1,Full,Out-degree,28576,62.0,2.048877,0.021178,2453,0.016456,0.004975,-1.582337,2.696662e-01,1377.109329,1.190181e-24,-4.295738,0.003377
2,Full,In-strength,58560,6.0,1.555974,0.003116,31826,0.035990,0.004975,-79.194928,3.232744e-19,56146.290641,0.000000e+00,-213.360941,0.000000


Esportati soltanto i risultati finali: C:\Moltbok\powerlaw_results_FULL.csv e C:\Moltbok\powerlaw_results_FULL.xlsx


332